# 0 Power Series, Differentiation, and Optimizers

This notebook introduces a basic workflow that we will reuse later for lattice optimization:

1. represent a function locally as a truncated power series,
2. extract derivatives using GTPSA,
3. pass those derivatives to an optimizer.

In the original Bmad/Tao workflow, one usually defines optimization data, variables, and a merit function, and Tao computes derivatives for the optimizer using finite differences. Here we first isolate the differentiation step and use **GTPSA** directly.


## 0.1 Setup

Run the next cell to load the packages used in this notebook.


In [8]:
using GTPSA
using Optim
using NLSolversBase


## 0.2 A local power series viewpoint

Given a scalar function $f(x,y)$ and an expansion point $(x_0,y_0)$, we write

$$
x = x_0 + \Delta x, \qquad y = y_0 + \Delta y.
$$

Then the function can be expanded locally as a Taylor series in $\Delta x$ and $\Delta y$. The constant term is the function value, the first-order terms give the gradient, and the second-order terms encode the Hessian.

GTPSA stores this information in a truncated power series object.


## 0.3 A toy objective function

We start with

$$
f(x,y) = (x-1)^2 + 2(y+0.3)^2 + 0.1xy.
$$

The goal is not to solve a hard optimization problem, but to show the workflow clearly.


## 0.4 First GTPSA example

In the next cell, we create a second-order TPS representation so that we can access the value, gradient, and Hessian.


In [9]:
# Keep terms through second order in two variables.
d = Descriptor(2, 2)

# TPS variables Δ[1], Δ[2]
Δ = vars(d)

# Expansion point
x0 = 2.0
y0 = -1.0

# Local variables
x = x0 + Δ[1]
y = y0 + Δ[2]

# Objective as a TPS expression
f = (x - 1)^2 + 2 * (y + 0.3)^2 + 0.1 * x * y

# Extract local information
value = f[0]
grad  = GTPSA.gradient(f)
H     = GTPSA.hessian(f)

println("value = ", value)
println("gradient = ", grad)
println("Hessian =")
println(H)


value = 1.78
gradient = [1.9, -2.5999999999999996]
Hessian =
[2.0 0.1; 0.1 4.0]


The meaning of the outputs is:

- `value`: the function value at $(x_0,y_0)$,
- `grad`: the gradient at $(x_0,y_0)$,
- `H`: the Hessian at $(x_0,y_0)$.

The important point is that these derivatives are read from the TPS representation itself.


## 0.5 Wrap this into a reusable function

In optimization, we evaluate the objective and derivatives at many trial points, so it is useful to wrap the calculation in a Julia function.


In [11]:
function value_gradient_hessian(xvec)
    d = Descriptor(2, 2)
    Δ = vars(d)

    x = xvec[1] + Δ[1]
    y = xvec[2] + Δ[2]

    f = (x - 1)^2 + 2 * (y + 0.3)^2 + 0.1 * x * y

    value = f[0]
    grad  = GTPSA.gradient(f)
    H     = GTPSA.hessian(f)

    return value, grad, H
end


value_gradient_hessian (generic function with 1 method)

In [12]:
val, g, H = value_gradient_hessian([2.0, -1.0])

println("value = ", val)
println("gradient = ", g)
println("Hessian =")
println(H)


value = 1.78
gradient = [1.9, -2.5999999999999996]
Hessian =
[2.0 0.1; 0.1 4.0]


## 0.6 Use the gradient in an optimizer

For a gradient-based optimizer, we usually need a function that returns the objective value and fills a gradient vector. Here we use `Optim.jl` with `LBFGS()`.


In [13]:
function value_gradient(xvec)
    d = Descriptor(2, 1)   # first order is enough for value + gradient
    Δ = vars(d)

    x = xvec[1] + Δ[1]
    y = xvec[2] + Δ[2]

    f = (x - 1)^2 + 2 * (y + 0.3)^2 + 0.1 * x * y

    value = f[0]
    grad  = GTPSA.gradient(f)

    return value, grad
end

function fg!(F, G, x)
    value, grad = value_gradient(x)

    if G !== nothing
        G .= grad
    end

    if F !== nothing
        return value
    end

    return nothing
end


fg! (generic function with 1 method)

In [15]:
x_init = [2.0, -1.0]

result = Optim.optimize(
    NLSolversBase.only_fg!(fg!),
    x_init,
    Optim.LBFGS()
)

println(result)
println("minimizer = ", Optim.minimizer(result))
println("minimum   = ", Optim.minimum(result))


 * Status: success

 * Candidate solution
    Final objective value:     -3.151439e-02

 * Found with
    Algorithm:     L-BFGS

 * Convergence measures
    |x - x'|               = 3.92e-01 ≰ 0.0e+00
    |x - x'|/|x'|          = 3.85e-01 ≰ 0.0e+00
    |f(x) - f(x')|         = 1.95e-01 ≰ 0.0e+00
    |f(x) - f(x')|/|f(x')| = 6.20e+00 ≰ 0.0e+00
    |g(x)|                 = 7.22e-16 ≤ 1.0e-08

 * Work counters
    Seconds run:   0  (vs limit Inf)
    Iterations:    2
    f(x) calls:    6
    ∇f(x) calls:   6

minimizer = [1.016270337922403, -0.3254067584480599]
minimum   = -0.03151439299123905


## 0.7 Why this matters for lattice design

This toy example has the same structure as a lattice optimization problem:

- the optimization variables will later become machine parameters such as quadrupole strengths,
- the objective will later become a merit function based on optics targets,
- GTPSA provides the local derivatives,
- the optimizer uses those derivatives to improve the parameters.

In the next notebook, we will apply exactly this pattern to a FODO cell.


## 0.8 Exercises

1. Replace the toy objective with

$$
f(x,y) = (x-0.5)^2 + (y+1.2)^2 + 0.2x^2 y^2
$$

and compute the value, gradient, and Hessian at $(1.0,-0.5)$.

2. Change the TPS order from 2 to 1 and verify that you can still access the gradient.

3. Try a different optimizer in `Optim.jl` and compare the number of iterations.

4. Add a penalty term $\lambda(x^2+y^2)$ and study how the minimizer changes as $\lambda$ varies.
